<a href="https://colab.research.google.com/github/Burakcelebi2003/english-vocabulary-test/blob/main/English_Vocabulary_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install streamlit numpy scipy pandas plotly

In [ ]:
#Color League - YDS Dil Tekrarı
%%writefile sabermetrics_gui.py
import streamlit as st
import pandas as pd
import numpy as np
import json
import os

# ============================================================
# 1. VERİTABANI (CSV) OKUMA
# ============================================================
@st.cache_data(ttl=5)
def load_vocabulary(file_path="Vocabulary.csv"):
    if not os.path.exists(file_path):
        return pd.DataFrame(columns=["id", "word", "meaning"])
    try:
        df = pd.read_csv(file_path, sep=";", header=None)
        if df.shape[1] >= 3:
            df = df.iloc[:, :3]
            df.columns = ["id", "word", "meaning"]
        elif df.shape[1] == 2:
            df.columns = ["word", "meaning"]
            df["id"] = range(1, len(df) + 1)

        df["word"] = df["word"].astype(str).str.strip()
        df["meaning"] = df["meaning"].astype(str).str.strip().str.lower()
        return df
    except Exception as e:
        st.error(f"Vocabulary.csv okuma hatası: {e}")
        return pd.DataFrame(columns=["id", "word", "meaning"])


# ============================================================
# 2. RENKLER LİGİ, TARİHÇE VE DİNAMİK FİKSTÜR MOTORU
# ============================================================
TEAMS = [
    "🔴Kırmızı", "🟠Turuncu", "🟡Sarı", "🟢Yeşil",
    "🔵Mavi", "🟣Mor", "💖Pembe", "⚫Siyah",
    "⚪Beyaz", "🩶Gri", "🌌Lacivert", "🩵Turkuaz"
]

DERBIES = [
    ("🟠Turuncu", "🔵Mavi"),
    ("⚫Siyah", "⚪Beyaz"),
    ("🔵Mavi", "🔴Kırmızı")
]

def is_derby(team_a, team_b):
    return (team_a, team_b) in DERBIES or (team_b, team_a) in DERBIES


# ============================================================
# 3. KALICI HAFIZA (JSON) YÖNETİMİ VE TARİHÇE
# ============================================================
MEMORY_FILE = "renkler_ligi_hafiza.json"

DEFAULT_HISTORICAL_CHAMPIONS = {
    "1986": "⚪Beyaz", "1987": "🔵Mavi", "1988": "🔵Mavi", "1989": "🔴Kırmızı", "1990": "🟠Turuncu",
    "1991": "🩶Gri", "1992": "🟠Turuncu", "1993": "🔵Mavi", "1994": "🔵Mavi", "1995": "🔵Mavi",
    "1996": "🟠Turuncu", "1997": "🟡Sarı", "1998": "🟡Sarı", "1999": "⚪Beyaz", "2000": "⚫Siyah",
    "2001": "🟠Turuncu", "2002": "🟠Turuncu", "2003": "🟠Turuncu", "2004": "🔵Mavi", "2005": "🔴Kırmızı",
    "2006": "🟠Turuncu", "2007": "🟠Turuncu", "2008": "🩵Turkuaz", "2009": "⚫Siyah", "2010": "⚪Beyaz",
    "2011": "🔴Kırmızı", "2012": "🟠Turuncu", "2013": "⚫Siyah", "2014": "💖Pembe", "2015": "🔴Kırmızı",
    "2016": "🔵Mavi", "2017": "🔵Mavi", "2018": "🔵Mavi", "2019": "🟠Turuncu", "2020": "🌌Lacivert",
    "2021": "🩶Gri", "2022": "🟠Turuncu", "2023": "🌌Lacivert", "2024": "🩶Gri", "2025": "⚫Siyah"
}

DEFAULT_CHAMPIONSHIP_COUNTS = {
    "🟠Turuncu": 11, "🔵Mavi": 9, "⚫Siyah": 4, "🔴Kırmızı": 4,
    "🩶Gri": 3, "⚪Beyaz": 3, "🌌Lacivert": 2, "🟡Sarı": 2,
    "💖Pembe": 1, "🩵Turkuaz": 1, "🟢Yeşil": 0, "🟣Mor": 0
}

def generate_fixture(teams):
    shuffled_teams = teams.copy()
    np.random.shuffle(shuffled_teams)
    n = len(shuffled_teams)
    team_indices = list(range(n))

    first_half = []
    for week in range(n - 1):
        week_matches = []
        for i in range(n // 2):
            home = shuffled_teams[team_indices[i]]
            away = shuffled_teams[team_indices[n - 1 - i]]
            if week % 2 == 1:
                week_matches.append((away, home))
            else:
                week_matches.append((home, away))
        first_half.append(week_matches)
        team_indices = [team_indices[0]] + [team_indices[-1]] + team_indices[1:-1]

    second_half = []
    for week_matches in first_half:
        reverse_matches = [(away, home) for home, away in week_matches]
        second_half.append(reverse_matches)

    return first_half + second_half


def init_league_state(force_reset=False):
    if force_reset or not os.path.exists(MEMORY_FILE):
        fixture = generate_fixture(TEAMS)
        standings = {team: {"O": 0, "G": 0, "B": 0, "M": 0, "AG": 0, "YG": 0, "P": 0} for team in TEAMS}

        state = {
            "season": 2026,
            "current_week": 0,
            "current_match_idx": 0,
            "standings": standings,
            "fixture": fixture,
            "match_results": {}, # "week_idx-match_idx": {"home": team, "away": team, "h_score": X, "a_score": Y}
            "match_in_progress": False,
            "match_words": [],
            "current_question_idx": 0,
            "home_score": 0,
            "away_score": 0,
            "last_answer_status": None,
            "historical_champions": DEFAULT_HISTORICAL_CHAMPIONS,
            "championship_counts": DEFAULT_CHAMPIONSHIP_COUNTS
        }
        save_league_state(state)
        return state
    else:
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
            if "historical_champions" not in data:
                data["historical_champions"] = DEFAULT_HISTORICAL_CHAMPIONS
            if "championship_counts" not in data:
                data["championship_counts"] = DEFAULT_CHAMPIONSHIP_COUNTS
            if "match_results" not in data:
                data["match_results"] = {}
            return data

def save_league_state(state):
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


# ============================================================
# 4. STREAMLIT ARAYÜZÜ VE OYUN MANTIĞI
# ============================================================
st.set_page_config(page_title="Color League - İngilizce Kelime Oyunu", layout="wide", page_icon="🏆")

if "league_state" not in st.session_state:
    st.session_state.league_state = init_league_state()

state = st.session_state.league_state
df_vocab = load_vocabulary("Vocabulary.csv")

# Lidere Göre Dinamik Arka Plan Rengi Belirleme
standings_df_temp = pd.DataFrame(state["standings"]).T
if not standings_df_temp.empty and standings_df_temp["P"].sum() > 0:
    standings_df_temp["Averaj"] = standings_df_temp["AG"] - standings_df_temp["YG"]
    leader_row = standings_df_temp.sort_values(by=["P", "Averaj", "AG"], ascending=False).iloc[0]
    leader_team = leader_row.name
else:
    leader_team = ""

# Renk Kodları Sözlüğü
TEAM_COLORS = {
    "🔴Kırmızı": "#ffe6e6", "🟠Turuncu": "#ffe8d6", "🟡Sarı": "#fff9db", "🟢Yeşil": "#e6f4ea",
    "🔵Mavi": "#e8f0fe", "🟣Mor": "#f3e8ff", "💖Pembe": "#fce8e6", "⚫Siyah": "#f1f3f4",
    "⚪Beyaz": "#ffffff", "🩶Gri": "#f8f9fa", "🌌Lacivert": "#e8eaf6", "🩵Turkuaz": "#e0f7fa"
}
bg_color = TEAM_COLORS.get(leader_team, "#ffffff")

st.markdown(
    f"""
    <style>
    .stApp {{
        background-color: {bg_color};
        transition: background-color 0.5s ease;
    }}
    </style>
    """,
    unsafe_allow_html=True
)

# Yan Menü (Sidebar)
st.sidebar.title("🏆Color League Paneli")
st.sidebar.markdown(f"📅Aktif Sezon: **{state.get('season', 2026)}**")
if leader_team:
    st.sidebar.markdown(f"👑Anlık Lider: **{leader_team}**")
st.sidebar.caption(f"📚Veritabanındaki Kelime Sayısı: **{len(df_vocab)}**")

if st.sidebar.button("🎮Ligi Sıfırla ve Yeni Sezon"):
    st.session_state.league_state = init_league_state(force_reset=True)
    st.rerun()

st.title(f"🌈Color League: {state.get('season', 2026)} İngilizce Kelime Şampiyonası")

# Tablar
tab_game, tab_standings, tab_fixture, tab_history = st.tabs(["⚔️Maç Ekranı", "📊Puan Tablosu", "📅Fikstür", "🏛️Lig Tarihçesi"])

# ------------------------------------------------------------
# TAB 1: MAÇ EKRANI
# ------------------------------------------------------------
with tab_game:
    if state["current_week"] >= 22:
        st.balloons()
        st.success(f"🎉{state['season']} SEZONU TAMAMLANDI!")

        df_final = pd.DataFrame(state["standings"]).T.sort_values(by=["P", "AG"], ascending=False)
        champion = df_final.index[0]
        st.header(f"🏆{state['season']} ŞAMPİYONU: {champion}")

        if st.button(f"🚀Yeni Sezona ({state['season'] + 1}) Geçiş Yap", use_container_width=True):
            # Sezon şampiyonunu tarihe ekle ve kupa sayısını güncelle
            season_str = str(state["season"])
            state["historical_champions"][season_str] = champion
            if champion in state["championship_counts"]:
                state["championship_counts"][champion] += 1
            else:
                state["championship_counts"][champion] = 1

            next_season = state["season"] + 1
            new_fixture = generate_fixture(TEAMS)
            standings = {team: {"O": 0, "G": 0, "B": 0, "M": 0, "AG": 0, "YG": 0, "P": 0} for team in TEAMS}

            state["season"] = next_season
            state["current_week"] = 0
            state["current_match_idx"] = 0
            state["standings"] = standings
            state["fixture"] = new_fixture
            state["match_results"] = {}
            state["match_in_progress"] = False
            save_league_state(state)
            st.rerun()
    else:
        current_week_num = state["current_week"] + 1
        current_match_num = state["current_match_idx"] + 1
        home_team, away_team = state["fixture"][state["current_week"]][state["current_match_idx"]]

        derby_badge = " 🔥**DERBİ MAÇI**🔥" if is_derby(home_team, away_team) else ""
        st.subheader(f"📅 {current_week_num}. Hafta | {current_match_num}. Maç{derby_badge}")

        if not state["match_in_progress"]:
            st.info(f"Sıradaki Maç: **{home_team}** vs **{away_team}**")
            if st.button("🚀Maçı Başlat", use_container_width=True):
                if len(df_vocab) < 20:
                    st.warning("⚠️Maç yapabilmek için Vocabulary.csv dosyasında en az 20 kelime olmalıdır!")
                else:
                    selected_sample = df_vocab.sample(n=20, replace=False).to_dict(orient="records")
                    state["match_words"] = selected_sample
                    state["match_in_progress"] = True
                    state["current_question_idx"] = 0
                    state["home_score"] = 0
                    state["away_score"] = 0
                    state["last_answer_status"] = None
                    save_league_state(state)
                    st.rerun()
        else:
            q_idx = state["current_question_idx"]

            if q_idx >= 20:
                st.success("🏁MAÇ SONA ERDİ!")
                h_score = state["home_score"]
                a_score = state["away_score"]

                st.markdown(f"### 🏆 **{home_team} {h_score} - {a_score} {away_team}**")

                if st.button("▶️Sonraki Maça Geç", use_container_width=True):
                    # Maç sonucunu kaydet
                    match_key = f"{state['current_week']}-{state['current_match_idx']}"
                    state["match_results"][match_key] = {
                        "home": home_team, "away": away_team, "h_score": h_score, "a_score": a_score
                    }

                    s = state["standings"]
                    s[home_team]["O"] += 1
                    s[away_team]["O"] += 1
                    s[home_team]["AG"] += h_score
                    s[home_team]["YG"] += a_score
                    s[away_team]["AG"] += a_score
                    s[away_team]["YG"] += h_score

                    if h_score > a_score:
                        s[home_team]["G"] += 1
                        s[home_team]["P"] += 3
                        s[away_team]["M"] += 1
                    elif a_score > h_score:
                        s[away_team]["G"] += 1
                        s[away_team]["P"] += 3
                        s[home_team]["M"] += 1
                    else:
                        s[home_team]["B"] += 1
                        s[away_team]["B"] += 1
                        s[home_team]["P"] += 1
                        s[away_team]["P"] += 1

                    if state["current_match_idx"] < 5:
                        state["current_match_idx"] += 1
                    else:
                        state["current_match_idx"] = 0
                        state["current_week"] += 1

                    state["match_in_progress"] = False
                    save_league_state(state)
                    st.rerun()
            else:
                active_team = home_team if q_idx % 2 == 0 else away_team
                current_word = state["match_words"][q_idx]

                sc1, sc2, sc3 = st.columns([2, 1, 2])
                sc1.metric(label=f"🏠{home_team}", value=f"{state['home_score']} Puan")
                sc2.metric(label="Soru Sırası", value=f"{q_idx + 1} / 20")
                sc3.metric(label=f"✈️{away_team}", value=f"{state['away_score']} Puan")

                st.markdown("---")
                st.markdown(f"### 🎯Sıradaki Takım: **{active_team}**")
                st.info(f"🔤İngilizce Kelime: **{current_word['word'].upper()}**")

                with st.form(key=f"vocab_form_{q_idx}"):
                    user_ans = st.text_input("Türkçe Anlamını Yazınız:", key=f"input_{q_idx}")
                    submit_btn = st.form_submit_button("Cevabı Gönder 🚀")

                if submit_btn:
                    clean_user_ans = user_ans.strip().lower()
                    correct_ans = str(current_word["meaning"]).strip().lower()

                    is_correct = (clean_user_ans == correct_ans) or (clean_user_ans in correct_ans.split(","))

                    if is_correct:
                        if q_idx % 2 == 0:
                            state["home_score"] += 1
                        else:
                            state["away_score"] += 1
                        state["last_answer_status"] = f"✅Doğru! (+1 Puan {active_team})"
                    else:
                        state["last_answer_status"] = f"❌Yanlış! Doğru Cevap: **{correct_ans}**"

                    state["current_question_idx"] += 1
                    save_league_state(state)
                    st.rerun()

                if state["last_answer_status"]:
                    st.caption(state["last_answer_status"])


# ------------------------------------------------------------
# TAB 2: PUAN TABLOSU
# ------------------------------------------------------------
with tab_standings:
    st.subheader(f"📊{state.get('season', 2026)} Puan Durumu")
    df_st = pd.DataFrame(state["standings"]).T
    df_st["Averaj"] = df_st["AG"] - df_st["YG"]
    df_st = df_st.sort_values(by=["P", "Averaj", "AG"], ascending=False)

    st.dataframe(df_st, use_container_width=True)

# ------------------------------------------------------------
# TAB 3: FİKSTÜR (Skorlu & Derbi Vurgulu)
# ------------------------------------------------------------
with tab_fixture:
    st.subheader(f"📅{state.get('season', 2026)} Haftalık Fikstür ve Skorlar")
    week_select = st.selectbox("Hafta Seçiniz:", range(1, 23), index=state["current_week"] if state["current_week"] < 22 else 21)

    selected_week_matches = state["fixture"][week_select - 1]

    fixture_display = []
    for m_idx, (h, a) in enumerate(selected_week_matches):
        match_key = f"{week_select - 1}-{m_idx}"
        derby_tag = " 🔥DERBİ" if is_derby(h, a) else ""

        if match_key in state["match_results"]:
            res = state["match_results"][match_key]
            score_str = f"{res['h_score']} - {res['a_score']}"
            status = "Oynandı"
        else:
            score_str = "Henüz Oynanmadı"
            status = "Bekliyor"

        fixture_display.append({
            "Ev Sahibi": h,
            "Skor": score_str,
            "Deplasman": a,
            "Durum": status + derby_tag
        })

    df_fix = pd.DataFrame(fixture_display)
    st.table(df_fix)

# ------------------------------------------------------------
# TAB 4: LİG TARİHÇESİ (MÜZE & GEÇMİŞ)
# ------------------------------------------------------------
with tab_history:
    st.subheader("🏛️Color League Onur Kürsüsü & Müze")

    col_h1, col_h2 = st.columns(2)

    with col_h1:
        st.markdown("### 🏆Toplam Şampiyonluklar")
        df_cups = pd.DataFrame(list(state["championship_counts"].items()), columns=["Takım", "Şampiyonluk Sayısı"])
        df_cups = df_cups.sort_values(by="Şampiyonluk Sayısı", ascending=False).reset_index(drop=True)
        st.dataframe(df_cups, use_container_width=True, hide_index=True)

    with col_h2:
        st.markdown("### 📜Sezon Sezon Şampiyonlar")
        df_hist = pd.DataFrame(list(state["historical_champions"].items()), columns=["Sezon", "Şampiyon Takım"])
        df_hist = df_hist.sort_values(by="Sezon", ascending=False).reset_index(drop=True)
        st.dataframe(df_hist, use_container_width=True, hide_index=True)

In [ ]:
import urllib
print("Şifre:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

In [ ]:
import subprocess
import os
import urllib.request
import platform
import re
import time
import threading

system = platform.system()
binary_name = "cloudflared.exe" if system == "Windows" else "cloudflared"

if not os.path.exists(binary_name):
    print("Cloudflare tünel aracı indiriliyor")
    if system == "Windows":
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
    else:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, binary_name)
    if system != "Windows":
        os.chmod(binary_name, 0o755)
    print("İndirme tamamlandı!")

def run_streamlit():
    subprocess.run(["streamlit", "run", "sabermetrics_gui.py", "--server.headless=true"])

streamlit_thread = threading.Thread(target=run_streamlit, daemon=True)
streamlit_thread.start()
time.sleep(3)

print("Cloudflare tüneli açılıyor")
cmd = [os.path.abspath(binary_name), "tunnel", "--url", "http://localhost:8501"]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

tunnel_url = None
for _ in range(15):
    line = process.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

if tunnel_url:
    print(f"\n==================================================")
    print(f"BAĞLANTI HAZIR!")
    print(f"{tunnel_url}")
    print(f"==================================================\n")
else:
    print("Tünel linki otomatik yakalanamadı")